# MEDDIAG — training from scratch on Kaggle

This notebook trains everything from zero: the FAISS search index, Stage-1, Stage-2, then the evaluation, experiments and plots. **Nothing is reused from older runs.** GPU memory is capped at 6 GB on purpose.

---

## ✅ How to run it (read this first)

### Part A — one-time setup (do this once)

**1. Upload the notebook.** On kaggle.com → **Code → New Notebook → File → Import Notebook**. Pick this file.
You can give the notebook any name. Nothing in it is tied to a particular account or computer: your Kaggle username and the notebook's own name are detected automatically.

**2. Turn on the GPU and internet.** In the right-hand panel (**Settings**):
- **Accelerator** → `GPU T4 x2`
- **Internet** → `On`

**3. Add your secret keys.** Top menu **Add-ons → Secrets**. Add each key below with *exactly* this name, and **tick the box** next to it. The values are in `secrets.md`.
- `HF_TOKEN`
- `KAGGLE_KEY`
- `SEMANTIC_SCHOLAR_API_KEY` (optional)

**4. Don't attach anything.** The model (LLaMA-3.2-3B) and all datasets download **from HuggingFace** automatically. Leave **Input** empty: no models, no datasets, no old checkpoints or FAISS.

Your HuggingFace account (the one `HF_TOKEN` belongs to) must have **accepted the licence** on the [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) page. Approval can take a few hours, so do it before the first run. The notebook checks this access in the first minutes and stops with a clear message if it's missing.

### Part B — start training (only once)

**5.** Click **Save Version** (top right) → pick **Save & Run All (Commit)** → click **Save**.

**6. That's it. Close the browser and switch the laptop OFF.** Training runs on Kaggle's computers, not yours.

**7. It keeps going by itself.** Each run lasts about 11 hours. At the end it saves everything and **starts the next run automatically**. This repeats until the log says **`ALL STAGES COMPLETE`**.

**8. To check progress** (optional, from any phone or PC): kaggle.com → **Your Work** → this notebook → **Versions** → click the newest one to see the live log.

> ❌ **Don't** press the ▶ Run buttons on cells to train. That mode stops when you close the browser. Always use **Save & Run All**.

**When do you need to click step 5 again?** Only if the chain stops. The log of the last run will say why:
- **Weekly GPU hours ran out** (Kaggle gives about 30 per week). Wait for the weekly reset, then do step 5 once. It continues where it left off.
- **"this run made no progress"**: something is broken, so the chain stopped itself instead of wasting GPU hours. Read the error in the log.

---

## 💾 Saving and resuming are automatic

You never save or load anything by hand:
- **While it runs:** everything (checkpoints, the FAISS index, logs, results) is saved to your Kaggle dataset **`meddiag-fresh-t4`** every 20 minutes, and once more just before the run stops.
- **When the next run starts:** it downloads the latest save and **continues exactly where it left off**. The very first run finds nothing, so it starts from zero.
- **If a run crashes or gets killed:** at most about 20 minutes of work is lost. The next run continues from the last save.

- **Starting the next run:** also automatic. It waits until the save has finished processing, then starts the next run, so the new run always loads the newest save.

## 📊 Where are the results?
Everything lives in your Kaggle dataset **`meddiag-fresh-t4`** (dataset page → ⋮ → Download):
- **`results/experiments/<date_time>/`**: all experiment results (Exp 1–9), each as `expN_results.json` plus its plots. This folder is kept separate from everything else.
- `lora_adapter/`, `cls_head.pt`, `projector_stage1.pt`: the trained model
- `eval_report_*.json` and `diagnostics/`: the evaluation report and plots

## ⏱️ How long it takes
Kaggle gives about **30 free GPU hours per week**, which is 2–3 runs. The whole training will take several runs, so expect 1–2 weeks. Progress carries over from run to run.

## 🆘 If something goes wrong
The log shows a plain-English error near the bottom. The usual causes:
- **"secret … is missing"** → step 3: the key wasn't added, or its box isn't ticked.
- **"No GPU"** → step 2: Accelerator isn't set.
- **"HuggingFace access problem"** → your `HF_TOKEN` is wrong, or that account hasn't been approved for LLaMA-3.2 yet (step 4).
- **"6 GB cap NOT enforced" / "no MEDDIAG_VRAM_CAP_GB support"** → the latest code isn't on GitHub yet. Push it, then run again.
- **Want to throw everything away and restart from zero?** In cell 1 set `FORCE_FRESH = True`, run once, then set it back to `False`.

In [ ]:
# 1. CONFIG ====================================================================
import os, time
RUN_START = time.time()                  # the time budget is measured from here
os.environ["HF_HUB_DISABLE_XET"] = "1"   # HF Xet CDN 403s from Kaggle -> classic LFS (must precede any HF import)

# --- Where checkpoints live between runs --------------------------------------
KAGGLE_USERNAME = ""                     # leave EMPTY: auto-detected from your KAGGLE_KEY in cell 3 (works on any account)
STATE_DATASET   = "meddiag-fresh-t4"     # created automatically in YOUR account on the first save
FORCE_FRESH     = False                  # True = ignore STATE_DATASET and restart from step 0 (it will be OVERWRITTEN)

# --- Hands-free chaining (laptop can stay OFF the whole time) ------------------
AUTO_CONTINUE   = True                   # at the end of a run, start the NEXT run of this notebook automatically
NOTEBOOK_SLUG   = ""                     # leave EMPTY: auto-detected (this notebook's URL name)

# --- Hard constraints -----------------------------------------------------------
VRAM_CAP_GB      = 6.0                   # project constraint: every process is capped to this much VRAM (T4 has 16)
TIME_BUDGET_H    = 11.0                  # stop training at this point; leaves ~1 h for the final upload before Kaggle's 12 h kill
UPLOAD_EVERY_MIN = 20                    # background checkpoint upload cadence

# --- Training recipe = the MEDDIAG paper (§3.3, §3.8, §3.9, Table 2) -----------
VISION_FINETUNE   = False                # §3.3: DenseNet-121 encoder FROZEN in both stages
VISION_AUGMENT    = False                # not part of the paper's method
CLS_ARCH          = "paper"              # §3.8: concat(3072 visual, 384 RAG) → 512 → GELU → 2
CLS_POOL          = "mean"               # §3.8: mean-pool visual tokens
CLS_NORMAL_WEIGHT = 2.0                  # §3.9: class-weighted CE, NORMAL = 2.0 ...
CLS_POS_WEIGHT    = 1.0                  #        ... ABNORMAL = 1.0
CLS_FOCAL_GAMMA   = 0.0                  # plain CE (no focal term)
CLS_LABEL_SMOOTH  = 0.0                  # plain CE (no label smoothing)
# λ_cls/λ_LM = 0.65/0.35, LoRA r16/α32/0.05 on q,k,v,o, LR 2e-4 (input_norm 2e-5),
# AdamW wd 0.01, warm-up 100 + cosine, clip 1.0, adversarial P = 0.20 are code defaults.
MAX_PAIRS_S2      = 4000
GRAD_ACCUM        = 4                    # "configurable accumulation" (§3.9); batch size 1 keeps it 6 GB-safe

REPO_URL    = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/vlm"

print(f"state dataset : <your account>/{STATE_DATASET}  (force_fresh={FORCE_FRESH})")
print(f"VRAM cap      : {VRAM_CAP_GB} GB   time budget: {TIME_BUDGET_H} h   upload every {UPLOAD_EVERY_MIN} min")
print(f"recipe (paper): encoder frozen={not VISION_FINETUNE} augment={VISION_AUGMENT} head={CLS_ARCH}/{CLS_POOL} "
      f"class weights N/A={CLS_NORMAL_WEIGHT}/{CLS_POS_WEIGHT} focal={CLS_FOCAL_GAMMA} "
      f"smooth={CLS_LABEL_SMOOTH} pairs_s2={MAX_PAIRS_S2} accum={GRAD_ACCUM}")

In [ ]:
# 2. PACKAGES ==================================================================
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("--upgrade", "kaggle")
pip("peft>=0.19.1", "bitsandbytes>=0.49.2", "accelerate>=1.13.0", "faiss-cpu==1.13.2",
    "sentence-transformers", "bert-score>=0.3.13", "torchxrayvision>=1.2.0", "python-dotenv")
print("Packages ready.")

In [ ]:
# 3. GPU, SECRETS & ENVIRONMENT ================================================
import os, json, glob
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # T4 x2 -> use one GPU only (6 GB budget is per-GPU)
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4, then Save & Run All again."
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name}  {_p.total_memory / 2**30:.1f} GiB total  -> capped to {VRAM_CAP_GB} GB per process")

secrets = UserSecretsClient()
def _secret(name, required=True):
    try:
        return secrets.get_secret(name).strip()
    except Exception as e:
        if required:
            raise RuntimeError(f"Kaggle secret '{name}' is missing. Add-ons -> Secrets -> add it AND tick it for this notebook.") from e
        return ""

os.environ["HF_TOKEN"] = _secret("HF_TOKEN")
_s2 = _secret("SEMANTIC_SCHOLAR_API_KEY", required=False)
if _s2: os.environ["SEMANTIC_SCHOLAR_API_KEY"] = _s2

# Kaggle API auth (used to download / upload STATE_DATASET). Works for ANY account:
# the username is never hard-coded — it is detected from the key.
import subprocess
_kkey = _secret("KAGGLE_KEY")
KAGGLE_USERNAME = (KAGGLE_USERNAME or _secret("KAGGLE_USERNAME", required=False)).strip()
_kd = Path.home() / ".kaggle"; _kd.mkdir(exist_ok=True)
if _kkey.startswith("KGAT_"):                 # new-style API token: identifies the account by itself
    os.environ["KAGGLE_API_TOKEN"] = _kkey
    (_kd / "access_token").write_text(_kkey); (_kd / "access_token").chmod(0o600)
else:                                         # legacy key (kaggle.json) needs the username alongside it
    if not KAGGLE_USERNAME:
        raise RuntimeError("Your KAGGLE_KEY is a legacy key, which needs the username too. Either create a new "
                           "API token (kaggle.com -> Settings -> API -> Create New Token, starts with KGAT_) "
                           "or add a secret KAGGLE_USERNAME with your Kaggle username.")
    os.environ["KAGGLE_KEY"] = _kkey
    (_kd / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _kkey}))
    (_kd / "kaggle.json").chmod(0o600)

def _detect_username():
    """Owner of this notebook (always saved in a Save & Run All run), else of any own dataset."""
    # --sort-by dateRun: in a Save & Run All run, THIS notebook is the most recently run one
    for cmd in (["kaggle", "kernels", "list", "--mine", "--sort-by", "dateRun", "--page-size", "1", "--csv"],
                ["kaggle", "datasets", "list", "--mine", "--csv"]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        for ln in (r.stdout or "").splitlines()[1:]:
            ref = ln.split(",")[0].strip()
            if "/" in ref:
                return ref.split("/")[0]
    return ""
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = _detect_username()
if not KAGGLE_USERNAME:
    raise RuntimeError("Could not detect your Kaggle username from KAGGLE_KEY. Check the key, or add a secret "
                       "KAGGLE_USERNAME with your Kaggle username.")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
print(f"Kaggle account: {KAGGLE_USERNAME}  (checkpoints -> {KAGGLE_USERNAME}/{STATE_DATASET})")

if not NOTEBOOK_SLUG:   # for auto-continue: this notebook = the most recently run one in this account
    _r = subprocess.run(["kaggle", "kernels", "list", "--mine", "--sort-by", "dateRun", "--page-size", "1", "--csv"],
                        capture_output=True, text=True)
    _refs = [ln.split(",")[0].strip() for ln in (_r.stdout or "").splitlines()[1:] if "/" in ln.split(",")[0]]
    NOTEBOOK_SLUG = _refs[0].split("/", 1)[1] if _refs else ""
print(f"This notebook : {KAGGLE_USERNAME}/{NOTEBOOK_SLUG or '(unknown - auto-continue will be skipped)'}")

# 6 GB constraint. MAX_VRAM_GB = where the LLM weights may be placed; VRAM_CAP_GB =
# hard allocator ceiling in EVERY child process (applied by src/config.py on import).
os.environ["MEDDIAG_MAX_VRAM_GB"]     = str(VRAM_CAP_GB)
os.environ["MEDDIAG_VRAM_CAP_GB"]     = str(VRAM_CAP_GB)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation under a tight cap
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONUNBUFFERED"]        = "1"
os.environ["MEDDIAG_MIMIC_REPO"]      = "power2004/mimic-cxr-dataset"

os.environ["MEDDIAG_VISION_AUGMENT"]  = "1" if VISION_AUGMENT else "0"
os.environ["MEDDIAG_VISION_FINETUNE"] = "1" if VISION_FINETUNE else "0"
os.environ["MEDDIAG_CLS_ARCH"]            = CLS_ARCH
os.environ["MEDDIAG_CLS_POOL"]            = CLS_POOL
os.environ["MEDDIAG_CLS_NORMAL_WEIGHT"]   = str(CLS_NORMAL_WEIGHT)
os.environ["MEDDIAG_CLS_POS_WEIGHT"]      = str(CLS_POS_WEIGHT)
os.environ["MEDDIAG_CLS_FOCAL_GAMMA"]     = str(CLS_FOCAL_GAMMA)
os.environ["MEDDIAG_CLS_LABEL_SMOOTHING"] = str(CLS_LABEL_SMOOTH)

# Model AND datasets come from HuggingFace (nothing attached on Kaggle).
LLM_REPO = "meta-llama/Llama-3.2-3B-Instruct"
os.environ["MEDDIAG_LLM_MODEL"] = LLM_REPO
if glob.glob("/kaggle/input/*"):
    print("WARNING: something is attached under Input. It isn't needed (everything comes from HuggingFace) "
          "and local copies of IU-Xray/Kermany would be used INSTEAD of HuggingFace. Remove it.")

from huggingface_hub import login, HfApi, snapshot_download
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
_api = HfApi(token=os.environ["HF_TOKEN"])

# --- HuggingFace preflight: fail in minute 1, not hour 5 ------------------------
# Each entry lists the mirrors the code tries in order; one reachable mirror is enough.
_HF_NEEDED = {   # name: (kind, [repo candidates], needed_for_training)
    "LLaMA-3.2-3B-Instruct (gated)": ("model",   [LLM_REPO], True),
    "MiniLM embedder":               ("model",   ["sentence-transformers/all-MiniLM-L6-v2"], True),
    # Paper: train + knowledge base on MIMIC-CXR only; the rest are evaluation sets.
    "MIMIC-CXR (train/KB/eval)":     ("dataset", ["power2004/mimic-cxr-dataset"], True),
    "IU-Xray w/ images (Exp 4A)":    ("dataset", ["ykumards/open-i", "MLforHealthcare/Indiana_University_Chest_X-ray_Collection",
                                                  "simwit/iu-xray"], False),
    "NIH ChestX-ray14 (Exp 1)":      ("dataset", ["BahaaEldin0/NIH-Chest-Xray-14", "Sohaibsoussi/NIH-Chest-X-ray-dataset-small",
                                                  "ud-medical/nih-chest-x-ray-dataset", "alkzar90/NIH-Chest-X-ray-dataset"], False),
    "PadChest (Exp 4B)":             ("dataset", ["JasonZZ0601/PadChest-GR-Xray", "agomberto/PadChest-multilabel"], False),
}
_fatal, _warn = [], []
for name, (kind, repos, for_training) in _HF_NEEDED.items():
    ok_repo, last_err = None, ""
    for repo in repos:
        try:
            (_api.model_info if kind == "model" else _api.dataset_info)(repo)
            ok_repo = repo; break
        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e).splitlines()[0][:150]}"
    print(f"  HF {'OK  ' if ok_repo else 'FAIL'} {name:<32} {ok_repo or last_err}")
    if not ok_repo: (_fatal if for_training else _warn).append(name)

# model_info can succeed on a gated repo without access; downloading a file proves access.
try:
    snapshot_download(LLM_REPO, token=os.environ["HF_TOKEN"],
                      allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
    print(f"  HF OK   {LLM_REPO} weights downloaded to the HF cache (reused by every pipeline step)")
except Exception as e:
    _fatal.append(f"{LLM_REPO} download ({type(e).__name__}: {str(e).splitlines()[0][:150]})")

if _warn:
    print(f"\n!! WARNING: experiment data unreachable: {_warn}. Training still runs, but those experiments "
          f"will FAIL at the end. Fix HuggingFace access before training finishes.")
if _fatal:
    raise RuntimeError(f"HuggingFace access problem for: {_fatal}. Check HF_TOKEN, and accept the licence on "
                       f"https://huggingface.co/{LLM_REPO} with the same account (approval can take hours).")
print("Secrets + env + HuggingFace access ready.")

In [ ]:
# 4. CLONE REPO ================================================================
import subprocess, os
from pathlib import Path
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:   # interactive re-run: update code, keep untracked models/ logs/
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--depth=1", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"])
os.chdir(REPO_DIR)
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

# The 6 GB hard cap lives in src/config.py. Refuse to train on an old checkout without it.
if "MEDDIAG_VRAM_CAP_GB" not in Path("src/config.py").read_text():
    raise RuntimeError("This repo checkout has no MEDDIAG_VRAM_CAP_GB support in src/config.py. "
                       "Push the latest commit to GitHub, then Save & Run All again.")

In [ ]:
# 5. RESTORE STATE FROM THE LATEST DATASET VERSION (auto-resume) ===============
# The dataset is the single source of truth. It's downloaded via the API (always
# the LATEST version), so an attached-but-stale input version can't cause a resume
# from an old step. First run / FORCE_FRESH -> nothing restored -> from scratch.
import os, re, json, shutil, zipfile, subprocess
from pathlib import Path
import torch

MODELS = Path(REPO_DIR) / "models"
LOGS   = Path(REPO_DIR) / "logs"
FAISS  = Path(REPO_DIR) / "faiss_index"
DL     = Path("/kaggle/working/_restore")

# Local workdir starts clean (commit runs are empty anyway; this also protects interactive re-runs).
for p in (MODELS, DL):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
LOGS.mkdir(exist_ok=True)
for f in (LOGS / ".pipeline_state", LOGS / ".pipeline.lock"):
    f.unlink(missing_ok=True)

def _expand_zips(root):
    """--dir-mode zip uploads each top-level folder as <name>.zip -> unpack to <name>/ (flatten a same-name nest)."""
    for z in list(Path(root).rglob("*.zip")):
        dst = z.with_suffix("")
        dst.mkdir(exist_ok=True)
        with zipfile.ZipFile(z) as zf: zf.extractall(dst)
        nested = dst / dst.name
        if nested.is_dir():
            for q in list(nested.iterdir()): shutil.move(str(q), str(dst / q.name))
            nested.rmdir()
        z.unlink()

def _first(root, name, want_dir=False):
    for r, ds, fs in os.walk(root):
        if name in (ds if want_dir else fs): return Path(r) / name
    return None

def _loads(path):
    try:
        torch.load(path, map_location="cpu", weights_only=False); return True
    except Exception as e:
        print(f"  [restore] unreadable, skipped: {path.name} ({type(e).__name__})"); return False

def _restore_faiss_from(root):
    d = _first(root, "faiss_index", want_dir=True)
    src = d if (d and (d / "index.faiss").exists()) else None
    if src is None:
        idx = _first(root, "index.faiss")
        if idx and (idx.parent / "meta.jsonl").exists(): src = idx.parent
    if src is None: return False
    if FAISS.exists(): shutil.rmtree(FAISS)
    FAISS.mkdir(parents=True)
    for fn in ("index.faiss", "meta.jsonl"): shutil.copy2(src / fn, FAISS / fn)
    return True

def _restore_bundle(root):
    got = {}
    pj = _first(root, "projector_stage1.pt")
    if pj and _loads(pj):
        shutil.copy2(pj, MODELS / pj.name); got["projector"] = True
    else:   # Stage-1 still in progress -> newest READABLE mid-checkpoint
        mids = sorted(((int(m.group(1)), Path(r) / f) for r, _, fs in os.walk(root) for f in fs
                       if (m := re.match(r"projector_step(\d+)\.pt$", f))), reverse=True)
        for step, p in mids:
            if _loads(p):
                shutil.copy2(p, MODELS / p.name); got["stage1_mid"] = step; break
    # Stage-2 checkpoints: copy all, then drop a newest one that is corrupt (killed mid-save),
    # because run_pipeline.sh would otherwise restart Stage-2 from step 0.
    s2 = sorted(((int(m.group(1)), Path(r) / d) for r, ds, _ in os.walk(root) for d in ds
                 if (m := re.match(r"lora_step(\d+)$", d))))
    for _, d in s2: shutil.copytree(d, MODELS / d.name, dirs_exist_ok=True)
    for step, _ in reversed(s2):
        ts = MODELS / f"lora_step{step}" / "train_state.pt"
        if ts.exists() and _loads(ts): got["stage2_step"] = step; break
        shutil.rmtree(MODELS / f"lora_step{step}")
    for name in ("cls_head.pt",):
        f = _first(root, name)
        if f: shutil.copy2(f, MODELS / name); got[name] = True
    for name in ("lora_adapter",):       # final Stage-2 adapter (present only once Stage-2 finished)
        d = _first(root, name, want_dir=True)
        if d: shutil.copytree(d, MODELS / name, dirs_exist_ok=True); got[name] = True
    got["faiss"] = _restore_faiss_from(root)
    for name in (".pipeline_state", "stage1.jsonl", "stage2.jsonl", "run.log"):
        f = _first(root, name)
        if f: shutil.copy2(f, LOGS / name)
    for f in Path(root).rglob("eval_report_*.json"): shutil.copy2(f, LOGS / f.name)
    for sub in ("results", "diagnostics"):   # results/experiments/<run>/ = Exp 1-9 outputs
        d = _first(root, sub, want_dir=True)
        if d: shutil.copytree(d, Path(REPO_DIR) / sub, dirs_exist_ok=True)
    return got

def dataset_exists():
    """True/False from the owner's dataset list. A missing dataset can 403 on download,
    so download errors alone can't tell 'first run' from 'API hiccup'."""
    r = subprocess.run(["kaggle", "datasets", "list", "--mine", "--search", STATE_DATASET, "--csv"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Kaggle API not usable (check the KAGGLE_KEY secret / KAGGLE_USERNAME): "
                           f"{((r.stderr or '') + (r.stdout or '')).strip()[-300:]}")
    want = f"{KAGGLE_USERNAME}/{STATE_DATASET}".lower()
    return any(ln.split(",")[0].strip().lower() == want for ln in r.stdout.splitlines())

got, source = {}, "none"
if FORCE_FRESH:
    print("[restore] FORCE_FRESH=True -> starting from scratch (dataset will be overwritten by the next upload)")
elif not dataset_exists():
    print(f"[restore] {KAGGLE_USERNAME}/{STATE_DATASET} does not exist yet -> FIRST RUN, from scratch")
else:
    r = subprocess.run(["kaggle", "datasets", "download", f"{KAGGLE_USERNAME}/{STATE_DATASET}",
                        "--unzip", "-p", str(DL)], capture_output=True, text=True)
    if r.returncode == 0 and any(DL.iterdir()):
        _expand_zips(DL); got = _restore_bundle(DL); source = "latest dataset version (API)"
    else:
        # Dataset exists but download failed: use the attached mount if present (may be an older version)
        mount = next((q for q in Path("/kaggle/input").rglob(STATE_DATASET) if q.is_dir()), None)
        if mount is None:
            raise RuntimeError(f"{STATE_DATASET} exists but could not be downloaded, and it isn't attached either. "
                               f"Refusing to silently restart from scratch. Kaggle CLI said: {msg}")
        print(f"[restore] API download failed ({msg}); using attached mount {mount} (may be an older version)")
        tmp = DL / "mount"; shutil.copytree(mount, tmp); _expand_zips(tmp)
        got = _restore_bundle(tmp); source = f"attached mount {mount}"

if not FAISS.joinpath("index.faiss").exists():
    # Fully fresh run: the FAISS index is NEVER taken from other/old datasets. It is
    # built by pipeline step 1 in this run, then saved to STATE_DATASET with the
    # checkpoints, so only THIS run's own index is restored in later sessions.
    print("[restore] no faiss_index saved by this run yet -> pipeline step 1 BUILDS it fresh (~1 h, then auto-saved)")
shutil.rmtree(DL, ignore_errors=True)

# Arch guard: a restored projector must be DenseNet-121 (1024-dim vision input).
_pj = next(iter(sorted(MODELS.glob("projector_*.pt"))), None)
if _pj is not None:
    _sd = torch.load(_pj, map_location="cpu", weights_only=False)
    _sd = _sd.get("projector", _sd.get("model", _sd)) if isinstance(_sd, dict) else _sd
    _vp = next((v for k, v in _sd.items() if str(k).endswith("vision_proj.weight")), None) if isinstance(_sd, dict) else None
    if _vp is not None and int(_vp.shape[1]) != 1024:
        raise RuntimeError(f"ARCH MISMATCH: {_pj.name} expects {int(_vp.shape[1])}-dim features; DenseNet-121 gives 1024.")

print(f"restore source : {source}")
print(f"restored       : {got or 'nothing (fresh start)'}")
print(f"faiss present  : {FAISS.joinpath('index.faiss').exists()}")

In [ ]:
# 6. SMOKE TEST + 6 GB CAP VERIFICATION (fails in seconds, not hours in) =======
import subprocess, sys, torch
from PIL import Image
from src.config import CONFIG
from src.vision import VisionEncoder
from src.projector import PerceiverResampler

assert CONFIG.vision_model_id.startswith("xrv:") and CONFIG.vision_hidden_dim == 1024, "encoder is not DenseNet-121"
ve = VisionEncoder(CONFIG)
px = ve.preprocess(Image.new("L", (256, 256), 128))
assert tuple(px.shape) == (1, 1, 224, 224), f"bad preprocess shape {tuple(px.shape)}"
tok = ve(px)
assert tok.shape[-1] == 1024, f"feature dim {tok.shape[-1]} != 1024"
proj = PerceiverResampler(vision_dim=CONFIG.vision_hidden_dim, llm_dim=CONFIG.llm_hidden_dim,
                          num_latents=CONFIG.num_visual_tokens,
                          num_heads=CONFIG.projector_num_heads, num_layers=CONFIG.projector_num_layers)
out = proj(tok.float())
assert tuple(out.shape) == (1, CONFIG.num_visual_tokens, CONFIG.llm_hidden_dim), f"projector out {tuple(out.shape)}"
del ve, proj, tok, out; torch.cuda.empty_cache()
print("[ok] DenseNet-121 -> projector shapes consistent")

# Prove the cap is live in a CHILD process (training runs as child processes):
# allocating VRAM_CAP_GB + 1 GB must fail with CUDA OOM.
_probe = (f"import src.config, torch; "
          f"torch.empty(int({VRAM_CAP_GB + 1} * 2**30), dtype=torch.uint8, device='cuda')")
r = subprocess.run([sys.executable, "-c", _probe], cwd=REPO_DIR, capture_output=True, text=True)
if r.returncode == 0 or "OutOfMemory" not in (r.stderr + r.stdout):
    raise RuntimeError(f"6 GB cap NOT enforced in child processes -- refusing to train.\n{r.stderr[-500:]}")
print(f"[ok] hard VRAM cap active: a {VRAM_CAP_GB + 1:.0f} GB allocation is rejected in child processes")

In [ ]:
# 7. PIPELINE STATE + RUN CONFIG ===============================================
# run_pipeline.sh --resume skips every step listed in logs/.pipeline_state.
# Rebuild it from the restored state, but only trust a marker if its artifact exists.
import re
from pathlib import Path
sp = LOGS / ".pipeline_state"
done = set(sp.read_text().split()) if sp.exists() else set()
done.add("step0")                                               # unit tests: not needed on Kaggle
done.discard("step1"); done.discard("step2"); done.discard("step3")
if (FAISS / "index.faiss").exists() and (FAISS / "meta.jsonl").exists(): done.add("step1")
if (MODELS / "projector_stage1.pt").exists():                   done.add("step2")
if (MODELS / "lora_adapter").is_dir() and (MODELS / "cls_head.pt").exists(): done.add("step3")
if "step3" not in done:                                         # eval needs the final adapter
    done -= {"step4", "step5", "step6", "step7"}
order = [f"step{i}" for i in range(8)]
sp.write_text("".join(s + "\n" for s in order if s in done))
print("already done:", [s for s in order if s in done])
print("to run      :", [s for s in order if s not in done] or "NOTHING (pipeline finished)")

sh = Path(REPO_DIR) / "run_pipeline.sh"; t = sh.read_text()
t = re.sub(r"GRAD_ACCUM=8\b", f"GRAD_ACCUM={GRAD_ACCUM}", t)          # non-smoke value only
t = re.sub(r"MAX_PAIRS_S2=4000\b", f"MAX_PAIRS_S2={MAX_PAIRS_S2}", t)
sh.write_text(t)
print(f"run_pipeline.sh: GRAD_ACCUM={GRAD_ACCUM} MAX_PAIRS_S2={MAX_PAIRS_S2}")

In [ ]:
# 8. TRAIN (time-boxed) + BACKGROUND CHECKPOINT UPLOADS ========================
import os, re, json, time, shutil, signal, subprocess, threading
from pathlib import Path

OUT = Path("/kaggle/working/_upload")
_lock = threading.Lock()
_FRESH_S = 90   # skip files written in the last 90 s (may be mid-save)

def _settled(p):
    p = Path(p)
    newest = max([p.stat().st_mtime] + [q.stat().st_mtime for q in p.rglob("*")]) if p.is_dir() else p.stat().st_mtime
    return time.time() - newest > _FRESH_S

def _fingerprint():
    items = []
    for p in list(MODELS.glob("*")) + [FAISS, LOGS / ".pipeline_state"]:
        if p.exists(): items.append((p.name, int(p.stat().st_mtime)))
    # experiment progress = number of finished expN_results.json files
    items.append(("experiments_done", len(list((Path(REPO_DIR) / "results").rglob("exp*_results.json")))))
    return tuple(sorted(items))

def package():
    if OUT.exists(): shutil.rmtree(OUT)
    OUT.mkdir(parents=True)
    n = 0
    for p in sorted(MODELS.iterdir()):
        try:   # checkpoint rotation may delete a folder while we copy it -> just skip that one
            if p.name == "lora_adapter_swa" or not _settled(p): continue
            (shutil.copytree if p.is_dir() else shutil.copy2)(p, OUT / p.name); n += 1
        except (FileNotFoundError, shutil.Error) as e:
            shutil.rmtree(OUT / p.name, ignore_errors=True)
            print(f"[upload] skipped {p.name} (changed during copy: {type(e).__name__})")
    if FAISS.joinpath("index.faiss").exists(): shutil.copytree(FAISS, OUT / "faiss_index")
    for name in (".pipeline_state", "stage1.jsonl", "stage2.jsonl", "run.log"):
        if (LOGS / name).exists(): shutil.copy2(LOGS / name, OUT / name)
    for f in LOGS.glob("eval_report_*.json"): shutil.copy2(f, OUT / f.name)
    for sub in ("results", "diagnostics"):   # results/ = its own folder in the dataset
        d = Path(REPO_DIR) / sub
        if d.is_dir() and any(d.rglob("*")): shutil.copytree(d, OUT / sub)
    return n

def upload(msg):
    with _lock:
        n = package()
        if n == 0 and not (OUT / "faiss_index").exists():
            print("[upload] nothing to upload yet"); return False
        (OUT / "dataset-metadata.json").write_text(json.dumps(
            {"title": STATE_DATASET, "id": f"{KAGGLE_USERNAME}/{STATE_DATASET}", "licenses": [{"name": "CC0-1.0"}]}))
        for attempt in range(3):
            try:
                exists = dataset_exists()
            except RuntimeError as e:
                print(f"[upload] attempt {attempt + 1}/3: {e}"); time.sleep(60); continue
            cmd = (["kaggle", "datasets", "version", "-p", str(OUT), "-m", msg, "--dir-mode", "zip"] if exists
                   else ["kaggle", "datasets", "create", "-p", str(OUT), "--dir-mode", "zip"])
            r = subprocess.run(cmd, capture_output=True, text=True)
            blob = (r.stdout or "") + (r.stderr or "")
            if r.returncode == 0 and ("error" not in blob.lower() or "being created" in blob.lower()):
                print(f"[upload] OK ({n} model items) -> {KAGGLE_USERNAME}/{STATE_DATASET}: {msg}"); return True
            print(f"[upload] attempt {attempt + 1}/3 failed: {blob.strip()[-300:]}")
            time.sleep(60)
        return False

stop_uploader = threading.Event()
def _uploader():
    last = _fingerprint()
    while not stop_uploader.wait(UPLOAD_EVERY_MIN * 60):
        try:   # never let one bad cycle kill the uploader thread
            fp = _fingerprint()
            if fp != last:
                hours = (time.time() - RUN_START) / 3600
                if upload(f"autosave at {hours:.1f} h"): last = fp
        except Exception as e:
            print(f"[upload] cycle failed, will retry next cycle: {type(e).__name__}: {e}")
threading.Thread(target=_uploader, daemon=True).start()

def wait_dataset_ready(timeout_s=1800):
    """A new dataset version takes a few minutes to process. The next run must not
    start before it's ready, or it would download the previous (older) version."""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        r = subprocess.run(["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{STATE_DATASET}"],
                           capture_output=True, text=True)
        if "ready" in (r.stdout or "").lower(): return True
        time.sleep(30)
    return False

def relaunch_self():
    """Start the next commit run of this notebook (same settings, inputs, secrets)."""
    d = Path("/kaggle/working/_relaunch")
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)
    if not NOTEBOOK_SLUG:
        print("[auto-continue] notebook name unknown - not relaunching"); return False
    ref = f"{KAGGLE_USERNAME}/{NOTEBOOK_SLUG}"
    r = subprocess.run(["kaggle", "kernels", "pull", ref, "-p", str(d), "-m"], capture_output=True, text=True)
    meta_f = d / "kernel-metadata.json"
    if r.returncode != 0 or not meta_f.exists():
        print(f"[auto-continue] could not pull {ref}: {((r.stderr or '') + (r.stdout or '')).strip()[-300:]}")
        return False
    # Safety: only ever relaunch a notebook that IS this pipeline (never some other notebook)
    if not any("AUTO_CONTINUE" in f.read_text(errors="ignore") and STATE_DATASET in f.read_text(errors="ignore")
               for f in d.glob("*.ipynb")):
        print(f"[auto-continue] {ref} is not this training notebook - not relaunching")
        return False
    meta = json.loads(meta_f.read_text())
    meta["enable_gpu"] = True; meta["enable_internet"] = True
    meta_f.write_text(json.dumps(meta, indent=2))
    for nbf in d.glob("*.ipynb"):   # never carry a one-off FORCE_FRESH=True into the next run
        nbf.write_text(re.sub(r"FORCE_FRESH(\s*)=(\s*)True", r"FORCE_FRESH\1=\2False", nbf.read_text()))
    r = subprocess.run(["kaggle", "kernels", "push", "-p", str(d)], capture_output=True, text=True)
    out = ((r.stdout or "") + (r.stderr or "")).strip()
    ok = r.returncode == 0 and "error" not in out.lower()
    print(f"[auto-continue] {'NEXT RUN STARTED' if ok else 'relaunch FAILED'}: {out[-300:]}")
    return ok

fp_start = _fingerprint()           # to detect "this run made no progress" (don't chain a broken run)
deadline = RUN_START + TIME_BUDGET_H * 3600
print(f"[train] budget ends in {(deadline - time.time()) / 3600:.2f} h; uploads every {UPLOAD_EVERY_MIN} min\n")
proc = subprocess.Popen(["bash", "run_pipeline.sh", "--resume"], cwd=REPO_DIR, env=os.environ.copy(),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        start_new_session=True)   # own process group -> clean kill of all children
def _pump():
    for line in proc.stdout: print(line, end="", flush=True)
threading.Thread(target=_pump, daemon=True).start()

timed_out = False
while proc.poll() is None:
    if time.time() > deadline:
        timed_out = True
        print(f"\n[train] TIME BUDGET ({TIME_BUDGET_H} h) reached -> stopping; progress since the last checkpoint is discarded")
        os.killpg(proc.pid, signal.SIGTERM)
        try: proc.wait(timeout=120)
        except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        break
    time.sleep(15)
time.sleep(2)
stop_uploader.set()
rc = proc.returncode
print(f"\n[train] run_pipeline.sh exited rc={rc} after {(time.time() - RUN_START) / 3600:.2f} h")

time.sleep(_FRESH_S + 5)          # let the last checkpoint count as settled
final_ok = upload("final upload of this run" + (" (time budget)" if timed_out else ""))

state = (LOGS / ".pipeline_state").read_text().split() if (LOGS / ".pipeline_state").exists() else []
complete = "step7" in state
progressed = _fingerprint() != fp_start
is_commit = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch"   # Save & Run All
chained = False
if complete:
    print("\n==================  ALL STAGES COMPLETE  ==================")
    print("Training, eval, experiments and plots are done. Nothing more to run.")
elif not AUTO_CONTINUE:
    print("\n[auto-continue] off (AUTO_CONTINUE=False)")
elif not is_commit:
    print("\n[auto-continue] skipped: this is an interactive session; only Save & Run All runs chain themselves")
elif not final_ok:
    print("\n[auto-continue] skipped: the final save failed, so the next run would lose this run's work")
elif not progressed:
    print("\n[auto-continue] STOPPED: this run made no progress. Something needs fixing (see log above); "
          "chaining would only burn GPU hours")
else:
    print("\n[auto-continue] waiting for the saved dataset version to finish processing ...")
    if wait_dataset_ready():
        chained = relaunch_self()
    else:
        print("[auto-continue] dataset still processing after 30 min; not chaining (next run could load an old save)")
if not complete and not chained:
    print(">>> NOT FINISHED YET and no next run was started. Click 'Save Version -> Save & Run All' "
          "to continue from the last save. If the weekly GPU quota ran out, do it after it resets.")
if rc != 0 and not timed_out:
    raise RuntimeError(f"run_pipeline.sh failed (rc={rc}); see the log above. Checkpoints were uploaded "
                       f"(final_ok={final_ok}), so fix the cause and Save & Run All again to resume.")
if not final_ok:
    raise RuntimeError("Final upload FAILED -- this run's progress is only in the notebook Output tab. "
                       "Create a dataset version from the output manually before relaunching.")
print(f"Checkpoints: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{STATE_DATASET}")

## Reference

**Monitor a run with the laptop off:** kaggle.com on any device → Your Work → this notebook → *Versions* → the running version shows the live log. The `stage1.jsonl` / `stage2.jsonl` logs include `vram_gb` (peak) per step. It must stay under 6.

**Weekly quota:** Kaggle gives about 30 GPU-hours per week, so plan on 2–3 runs a week. Each run picks up where the last one ended.

**Download the weights** (when finished): dataset page → ⋮ → Download, or
```
kaggle datasets download <your-kaggle-username>/meddiag-fresh-t4 --unzip -p ./out
```

**If something goes wrong**
- `6 GB cap NOT enforced`: the GitHub repo is missing the `MEDDIAG_VRAM_CAP_GB` change in `src/config.py`. Push it.
- CUDA OOM during training: the model genuinely exceeds 6 GB at that step. Lower `GRAD_ACCUM` doesn't help (batch is 1). Check which stage OOMs before changing anything.
- Start over completely: set `FORCE_FRESH = True` for one run, then set it back to `False`.